In [ ]:
# cell 1
# Install runtime dependencies.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy accelerate safetensors huggingface_hub

# Remove optional packages that may break imports in some Colab/vLLM environments.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Gemma 4 needs recent Transformers support.
!uv pip install --system -U "transformers>=5.5.0"

# Recent vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 61 packages in 82ms
Prepared 10 packages in 0.36ms
Uninstalled 10 packages in 125ms
Installed 10 packages in 115ms
 - cuda-toolkit==13.0.2
 + cuda-toolkit==13.0.3.0
 - numpy==2.3.5
 + numpy==2.5.1
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - nvidia-nvjitlink==13.0.88
 + nvidia-nvjitlink==13.3.33
 - setuptools==80.10.2
 + setuptools==83.0.0
 - torch==2.11.0+cu130
 + torch==2.13.0
 - triton==3.6.0
 + triton==3.7.1
Using Python 3.12.13 environment at: /usr
Uninstalled 3 packages in 49ms
 - torchaudio==2.11.0+cu130
 - torchcodec==0.15.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 67ms
Checked 27 packages in 0.28ms
Using Python 3.12.13 environment at: /usr
Resolved 191 packages in 6.53s
Pre

In [ ]:
#cell 2
# Imports and global config.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback

from pathlib import Path
from tqdm.auto import tqdm
from openai import OpenAI
from jsonschema import validate

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DATASET = "2wikimultihopqa"

LLM_MODEL_NAME = "google/gemma-4-26B-A4B-it"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Keep the same Gemma4 runtime parameters as the original notebook.
MAX_MODEL_LEN = 131072
GPU_MEMORY_UTILIZATION = 0.92
MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

SERVER_LOG_PATH = Path("/content/vllm_gemma4_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gemma4_answer_server.pid")

# Local runtime copy directory.
LOCAL_RUNTIME_DIR = Path("/content/final_project_copy")
LOCAL_ABLATION_DIR = LOCAL_RUNTIME_DIR / "ablation"
LOCAL_ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# Google Drive project paths.
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
DRIVE_ABLATION_DIR = DRIVE_PROJECT_DIR / "ablation"

# Same evidence filename for all ablation runs.
EVIDENCE_FILENAME = (
    "2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json"
)

# Gemma4 answer filename.
ANSWER_FILENAME = "2wikimultihopqa_answer_gemma4.json"

# Run only the three new ablation experiments independently.
ABLATION_NAMES = [
    "without_dense_faiss",
    "without_bm25_token",
    "without_bm25_3char",
]


def make_ablation_config(ablation_name):
    # Build all paths for one independent ablation run.
    drive_evidence_path = (
        DRIVE_ABLATION_DIR
        / ablation_name
        / EVIDENCE_FILENAME
    )

    local_evidence_path = (
        LOCAL_ABLATION_DIR
        / ablation_name
        / EVIDENCE_FILENAME
    )

    drive_answer_dir = (
        DRIVE_ABLATION_DIR
        / ablation_name
        / "answer"
    )

    drive_answer_path = (
        drive_answer_dir
        / ANSWER_FILENAME
    )

    return {
        "name": ablation_name,
        "drive_evidence_path": drive_evidence_path,
        "local_evidence_path": local_evidence_path,
        "drive_answer_dir": drive_answer_dir,
        "drive_answer_path": drive_answer_path,
    }


ABLATION_RUNS = [
    make_ablation_config(name)
    for name in ABLATION_NAMES
]

ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None  # None means all records.

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# Keep the same Gemma4 answer-generation parameters as the original notebook.
ANSWER_MAX_TOKENS = 32768
ANSWER_RETRY_MAX_TOKENS = 65536
LOW_COMPLETION_TOKENS_THRESHOLD = 2048

print("Model:", LLM_MODEL_NAME)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)

print("\nAblation runs:")

for run in ABLATION_RUNS:
    print("=" * 100)
    print("Name:", run["name"])
    print("Drive evidence path:", run["drive_evidence_path"])
    print("Local evidence path:", run["local_evidence_path"])
    print("Answer output path:", run["drive_answer_path"])

Model: google/gemma-4-26B-A4B-it
Max model len: 131072
GPU memory utilization: 0.92

Ablation runs:
Name: without_dense_faiss
Drive evidence path: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Local evidence path: /content/final_project_copy/ablation/without_dense_faiss/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Answer output path: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/answer/2wikimultihopqa_answer_gemma4.json
Name: without_bm25_token
Drive evidence path: /content/drive/MyDrive/final_project/ablation/without_bm25_token/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Local evidence path: /content/final_project_copy/ablation/without_bm25_token/2wikimultihopqa_dev_2020wiki_1000_traversal_evidence.json
Answer output path: /content/drive/MyDrive/final_project/ablation/without_bm25_token/answer/2wikimultihopqa_answer_gemma4.json
Name: without_bm25_3char
Drive evi

In [ ]:
#cell 3
# Mount Google Drive and copy the three new ablation evidence files
# to the local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)


def file_is_same_size(src: Path, dst: Path) -> bool:
    # Check whether the local copy is complete.
    return (
        dst.exists()
        and dst.stat().st_size == src.stat().st_size
    )


def copy_file_to_local(src: Path, dst: Path) -> None:
    # Copy with a temporary file to avoid partial local copies.
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print("Local evidence copy already exists:", dst)
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


for run in ABLATION_RUNS:
    name = run["name"]
    drive_evidence_path = run["drive_evidence_path"]
    local_evidence_path = run["local_evidence_path"]
    drive_answer_dir = run["drive_answer_dir"]

    # Check that the evidence file exists in Google Drive.
    if not drive_evidence_path.exists():
        raise FileNotFoundError(
            f"Evidence file not found for ablation "
            f"'{name}': {drive_evidence_path}"
        )

    # The answer directory already exists and is not recreated here.
    if not drive_answer_dir.is_dir():
        raise FileNotFoundError(
            f"Answer directory not found for ablation "
            f"'{name}': {drive_answer_dir}"
        )

    # Copy the evidence file from Google Drive to local Colab storage.
    copy_file_to_local(
        drive_evidence_path,
        local_evidence_path,
    )

    print("=" * 100)
    print("Ablation:", name)
    print("Evidence copied to local disk.")
    print(
        "Local evidence size MB:",
        local_evidence_path.stat().st_size / (1024 ** 2),
    )
    print("Answer directory:", drive_answer_dir)
    print("Answer file:", run["drive_answer_path"])

Mounted at /content/drive
Ablation: without_dense_faiss
Evidence copied to local disk.
Local evidence size MB: 10.776281356811523
Answer directory: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/answer
Answer file: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/answer/2wikimultihopqa_answer_gemma4.json
Ablation: without_bm25_token
Evidence copied to local disk.
Local evidence size MB: 10.801043510437012
Answer directory: /content/drive/MyDrive/final_project/ablation/without_bm25_token/answer
Answer file: /content/drive/MyDrive/final_project/ablation/without_bm25_token/answer/2wikimultihopqa_answer_gemma4.json
Ablation: without_bm25_3char
Evidence copied to local disk.
Local evidence size MB: 10.78447151184082
Answer directory: /content/drive/MyDrive/final_project/ablation/without_bm25_3char/answer
Answer file: /content/drive/MyDrive/final_project/ablation/without_bm25_3char/answer/2wikimultihopqa_answer_gemma4.json


In [ ]:
# cell 4
# Validate all ablation traversal evidence files.
# Load the first ablation only for prompt preview compatibility with cell 8.

required_keys = ["type", "question", "answer", "evidence_chunk", "evidence_path"]

def load_evidence_records(path: Path):
    # Load one traversal evidence JSON file.
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    if not isinstance(records, list):
        raise RuntimeError(f"Evidence JSON must be a list of records: {path}")

    return records

def validate_evidence_records(records, ablation_name, check_first_n=5):
    # Validate required keys in the first few records.
    for i, rec in enumerate(records[:check_first_n]):
        missing = [k for k in required_keys if k not in rec]
        if missing:
            print(f"Warning: ablation '{ablation_name}', record {i} missing keys:", missing)

for run in ABLATION_RUNS:
    name = run["name"]
    local_evidence_path = run["local_evidence_path"]

    records = load_evidence_records(local_evidence_path)
    validate_evidence_records(records, name)

    run["num_records"] = len(records)

    print("=" * 100)
    print("Ablation:", name)
    print("Number of evidence records:", len(records))
    print("First source_index:", records[0].get("source_index", 0))
    print("First question:", records[0].get("question"))
    print("First GT answer:", records[0].get("answer"))
    print("First chunks:", len(records[0].get("evidence_chunk", [])))
    print("First paths:", len(records[0].get("evidence_path", [])))

    del records

gc.collect()

# Keep this variable only so the original prompt-preview cell can run unchanged.
preview_run = ABLATION_RUNS[0]
evidence_records = load_evidence_records(preview_run["local_evidence_path"])

print("=" * 100)
print("Preview evidence loaded from ablation:", preview_run["name"])
print("Preview records:", len(evidence_records))

Ablation: without_dense_faiss
Number of evidence records: 1000
First source_index: 0
First question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?
First GT answer: no
First chunks: 6
First paths: 5
Ablation: without_bm25_token
Number of evidence records: 1000
First source_index: 0
First question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?
First GT answer: no
First chunks: 6
First paths: 5
Ablation: without_bm25_3char
Number of evidence records: 1000
First source_index: 0
First question: Are North Marion High School (Oregon) and Seoul High School both located in the same country?
First GT answer: no
First chunks: 6
First paths: 5
Preview evidence loaded from ablation: without_dense_faiss
Preview records: 1000


In [ ]:
# cell 5
# Start Gemma 4 vLLM server in thinking mode.

def kill_process_tree(pid):
    # Kill a process and all children.
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_or_download_gemma4_chat_template():
    # Find or download the Gemma 4 vLLM chat template.
    template_name = "tool_chat_template_gemma4.jinja"
    local_template = Path("/content") / template_name

    if local_template.exists() and local_template.stat().st_size > 0:
        return local_template

    search_roots = []

    try:
        import vllm as vllm_pkg
        vllm_path = Path(vllm_pkg.__file__).resolve()
        search_roots.extend([
            vllm_path.parent,
            vllm_path.parent.parent,
        ])
    except Exception:
        pass

    search_roots.extend([
        Path("/usr/local/lib/python3.12/dist-packages"),
        Path("/usr/local/lib/python3.12/site-packages"),
        Path("/usr/lib/python3.12/site-packages"),
        Path("/content"),
    ])

    for root in search_roots:
        if not root.exists():
            continue

        try:
            matches = sorted(root.rglob(template_name))
        except Exception:
            matches = []

        for match in matches:
            if match.exists() and match.stat().st_size > 0:
                print("Found Gemma 4 chat template:", match)
                return match

    # Fallback for pip installs where examples are not packaged.
    try:
        import requests

        url = (
            "https://raw.githubusercontent.com/vllm-project/vllm/main/"
            "examples/tool_chat_template_gemma4.jinja"
        )
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        local_template.write_text(r.text, encoding="utf-8")

        if local_template.exists() and local_template.stat().st_size > 0:
            print("Downloaded Gemma 4 chat template:", local_template)
            return local_template

    except Exception as e:
        print("Warning: could not download Gemma 4 chat template:", repr(e))

    return None

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

gemma4_chat_template_path = find_or_download_gemma4_chat_template()

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only QA workload.
    "--language-model-only",
    "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',

    # Gemma 4 thinking / tool protocol support.
    "--reasoning-parser", "gemma4",
    "--tool-call-parser", "gemma4",
    "--enable-auto-tool-choice",
    "--default-chat-template-kwargs", '{"enable_thinking": true}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",

    # Force Triton MoE backend to avoid FlashInfer CUTLASS MoE JIT crash on this vLLM nightly.
    "--moe-backend", "triton",

    "--trust-remote-code",
]

if gemma4_chat_template_path is not None:
    cmd.extend(["--chat-template", str(gemma4_chat_template_path)])
else:
    print("Warning: running without explicit Gemma 4 tool chat template.")

server_env = os.environ.copy()

# Blackwell / CUDA 13 fixes.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Downloaded Gemma 4 chat template: /content/tool_chat_template_gemma4.jinja
Command:
vllm serve google/gemma-4-26B-A4B-it --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.92 --language-model-only --limit-mm-per-prompt '{"image": 0, "audio": 0}' --reasoning-parser gemma4 --tool-call-parser gemma4 --enable-auto-tool-choice --default-chat-template-kwargs '{"enable_thinking": true}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --moe-backend triton --trust-remote-code --chat-template /content/tool_chat_template_gemma4.jinja

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 8610
Log: /content/vllm_gemma4_answer_server.log


In [ ]:
# cell 6
# Wait for vLLM server and create OpenAI-compatible client.

import requests

def tail_log(path, n=80):
    # Read last log lines.
    path = Path(path)
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

print("OpenAI-compatible client is ready.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=9086) INFO 07-15 21:21:09 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=9086) INFO 07-15 21:21:09 [parallel_state.py:1607] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:48407 backend=nccl
(EngineCore pid=9086) INFO 07-15 21:21:09 [parallel_state.py:1942] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(EngineCore pid=9086) INFO 07-15 21:21:10 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=9086) INFO 07-15 21:21:10 [gpu_model_runner.py:5236] Starting to load model google/gemma-4-26B-A4B-it...
(EngineCore pid=9086) INFO 07-15 21:21:10 [vllm.py:1090] Asynchronous scheduling is enabled.
(EngineCore pid=9086) INFO 07-15 21:

In [ ]:
# cell 7
# Answer schema and prompt.

ANSWER_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "response": {
            "type": "string",
            "minLength": 1
        }
    },
    "required": ["response"]
}

ANSWER_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "hotpotqa_answer",
        "schema": ANSWER_SCHEMA,
    },
}

ANSWER_SYSTEM_PROMPT = """
You are a careful and highly capable multi-hop open-domain question answering agent.

You will receive:
1. A question.
2. Final evidence chunks retrieved by a knowledge-graph traversal agent.
3. Final evidence paths retrieved from the same knowledge graph.

Knowledge graph structure:
- Node types:
  1. entity nodes
     - Entity nodes represent normalized entities.
  2. chunk nodes
     - Chunk nodes represent Wikipedia text chunks.

- Edge types:
  1. entity --relation--> entity
     - Directed edge from one entity to another entity.
     - The relation text is a short natural-language sentence describing the connection.
     - Each relation edge has a relation_id and a source chunk_id.
  2. entity --fact--> chunk
     - Directed edge from an entity to a Wikipedia chunk.
     - The fact text describes what information the chunk contains about that entity.
     - Each fact edge has a fact_id and a chunk_id.

Traversal summary:
- A traversal agent explored the knowledge graph to collect evidence for the question.
- The selected chunks may contain bridge evidence, comparison evidence, answer-bearing evidence, or irrelevant distractors.
- The selected paths may help connect entities across multiple hops.
- Some chunks and paths can be noisy or misleading. Treat them as possible distractors, not guaranteed evidence.
- Prefer explicit evidence from chunks, but use paths to guide multi-hop reasoning when they help connect facts.

Your task:
Answer the question using only the provided evidence chunks and evidence paths.

Reasoning instructions:
- In your thinking phase, carefully analyze the question, evidence chunks, and evidence paths.
- Identify the relevant entities and the required reasoning type: bridge, comparison, or another multi-hop pattern.
- Connect facts across multiple chunks when needed.
- Use evidence paths as supporting signals for entity connections, but do not treat a path as stronger than explicit chunk text.
- Ignore distractor chunks or paths that are not needed for the answer.
- Do not use external knowledge.
- Do not guess beyond the provided evidence.

Answering rules:
1. Your primary goal is to synthesize the answer from the provided evidence.
2. Do not give up easily. Many questions require combining evidence from two or more chunks.
3. Only if the provided evidence is truly insufficient, contradictory, or does not support any answer after careful multi-hop reasoning, the final response must be exactly:
   Information not available
4. The final answer should be extremely concise.
5. Prefer a short answer phrase of 1 to 10 words when possible.
6. Never exceed 18 words unless the exact required response is Information not available.
7. Do not include explanations, citations, reasoning traces, or extra text in the final answer string.

Output format:
- You may reason during the model's thinking phase.
- After reasoning is complete, the final assistant content must contain exactly one valid JSON object.
- The JSON object must match this schema:
{
  "response": "..."
}
- Do not wrap the JSON in markdown.
- Do not add any text before or after the JSON.
""".strip()

In [ ]:
# cell 8
# Prompt formatting helpers.

def clean_text_for_prompt(text):
    # Preserve text content, only normalize Python None.
    if text is None:
        return ""
    return str(text)

def format_evidence_chunks(chunks):
    # Format all chunks without truncation.
    if not chunks:
        return "No evidence chunks provided."

    blocks = []

    for i, chunk in enumerate(chunks, start=1):
        title = clean_text_for_prompt(chunk.get("title", ""))
        text = clean_text_for_prompt(chunk.get("text", ""))

        block = (
            f"[Chunk {i}]\n"
            f"Title: {title}\n"
            f"Text:\n{text}"
        )
        blocks.append(block)

    return "\n\n".join(blocks)

def format_evidence_paths(paths):
    # Format all paths as p1, p2, ...
    if not paths:
        return "No evidence paths provided."

    lines = []

    for i, path_item in enumerate(paths, start=1):
        path_text = clean_text_for_prompt(path_item.get("path", ""))
        lines.append(f"p{i}: {path_text}")

    return "\n".join(lines)

def build_answer_prompt(record):
    # Build final QA prompt.
    question = clean_text_for_prompt(record.get("question", ""))

    chunks_text = format_evidence_chunks(record.get("evidence_chunk", []))
    paths_text = format_evidence_paths(record.get("evidence_path", []))

    prompt = f"""
Question:
{question}

Evidence chunks:
{chunks_text}

Evidence paths:
{paths_text}

Now answer the question. You may reason in the thinking phase, then output the final answer as one valid JSON object only.
""".strip()

    return prompt

# Preview one prompt.
preview_prompt = build_answer_prompt(evidence_records[0])
print(preview_prompt[:4000])
print("\nPrompt preview length in characters:", len(preview_prompt))

Question:
Are North Marion High School (Oregon) and Seoul High School both located in the same country?

Evidence chunks:
[Chunk 1]
Title: North Marion High School (Oregon)
Text:
North Marion High School is a public high school in Aurora, Oregon, United States. The school is part of the North Marion School District with all four schools being located on the same campus. The school draws students from the cities of Aurora, Hubbard, and Donald as well as the communities of Broadacres and Butteville.
Academics: The school earned a 4 (out of 5) in its report card grading for the 2013-2014 school year, meaning more than 70 percent of students met or exceeded standards on the Oregon Assessment of Knowledge and Skills. North Marion High School's completion rate was 94.8 percent, and its four-year graduation rate was 78.6 percent.
OAKS testing scores: 2013-2014 North Marion High School test scores: Reading: 95.7% passed or exceeded State Average: 85.6% Writing: 74.8% passed or exceeded State A

In [ ]:
# cell 9
# JSON parsing and answer cleanup.

def strip_code_fence(text):
    # Remove markdown code fences.
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return text.strip()

def strip_think_blocks(text):
    # Remove raw thinking blocks if returned.
    text = text or ""

    # Qwen / generic thinking tags.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)

    # Gemma 4 thinking channel tags.
    text = re.sub(
        r"<\|channel\>thought\s*.*?<channel\|>",
        "",
        text,
        flags=re.DOTALL,
    )

    return text.strip()

def extract_json_object(text):
    # Extract first JSON object from model output.
    text = strip_code_fence(strip_think_blocks(text))
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found. Raw text:\n{text[:2000]}")

    return text[start:end + 1]

def normalize_answer_text(answer):
    # Keep answer short and clean.
    answer = strip_think_blocks(answer)
    answer = strip_code_fence(answer)
    answer = re.sub(r"\s+", " ", answer).strip()

    if not answer:
        return "Information not available"

    # Remove accidental surrounding quotes.
    if len(answer) >= 2 and answer[0] == answer[-1] and answer[0] in ['"', "'"]:
        answer = answer[1:-1].strip()

    # Enforce exact fallback text.
    if answer.lower() in {
        "not available",
        "information unavailable",
        "unknown",
        "cannot determine",
        "not enough information",
        "insufficient information",
    }:
        return "Information not available"

    return answer

def parse_answer_json(content):
    # Parse answer JSON and validate schema.
    json_text = extract_json_object(content)
    parsed = json.loads(json_text)
    validate(instance=parsed, schema=ANSWER_SCHEMA)

    response = normalize_answer_text(parsed["response"])
    parsed["response"] = response

    return parsed

In [ ]:
#cell 10
# LLM answer function with retry only for invalid or broken JSON.

LLM_SAMPLING_KWARGS = {
    "temperature": 1.0,
    "top_p": 0.95,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 64,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": True,
    },
}

def get_completion_tokens_from_usage(usage):
    # Read completion token count if vLLM/OpenAI-compatible usage provides it.
    if usage is None:
        return None

    try:
        usage_dict = usage.model_dump()
    except Exception:
        try:
            usage_dict = dict(usage)
        except Exception:
            return None

    return usage_dict.get("completion_tokens")


def get_reasoning_from_message(msg):
    # Read Gemma/Qwen/vLLM reasoning field robustly.
    for attr in ["reasoning_content", "reasoning"]:
        value = getattr(msg, attr, None)
        if value:
            return value
    return None


def should_retry_after_parse_error(response, content):
    # Retry only when the final output is not valid JSON.
    usage = response.usage
    completion_tokens = get_completion_tokens_from_usage(usage)

    finish_reason = None
    try:
        finish_reason = response.choices[0].finish_reason
    except Exception:
        pass

    low_completion = (
        completion_tokens is not None
        and completion_tokens < LOW_COMPLETION_TOKENS_THRESHOLD
    )

    hit_length_limit = finish_reason == "length"

    return {
        "retry": True,
        "completion_tokens": completion_tokens,
        "low_completion": low_completion,
        "finish_reason": finish_reason,
        "hit_length_limit": hit_length_limit,
    }


def create_chat_completion_with_optional_schema(**kwargs):
    # Use guided JSON schema when available; fallback if unsupported.
    try:
        return client.chat.completions.create(
            **kwargs,
            response_format=ANSWER_RESPONSE_FORMAT,
        )
    except Exception as e:
        err = str(e).lower()

        schema_error_signals = [
            "response_format",
            "guided",
            "json_schema",
            "schema",
            "not supported",
            "unsupported",
        ]

        if any(s in err for s in schema_error_signals):
            print("Warning: response_format failed; retrying without guided schema.")
            print("Schema error:", repr(e))
            return client.chat.completions.create(**kwargs)

        raise


def call_llm_answer(prompt, max_retries=2):
    # Call thinking-enabled Gemma 4 and parse short JSON answer.
    # Retry only if JSON parsing/validation fails.
    # Do not retry valid Information not available responses.
    last_error = None
    last_content = None
    last_reasoning = None
    last_retry_info = None

    messages = [
        {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    for attempt in range(max_retries + 1):
        current_messages = messages

        # First attempt uses normal thinking/output budget.
        # Retries use larger budget but keep repair context compact.
        current_max_tokens = ANSWER_MAX_TOKENS if attempt == 0 else ANSWER_RETRY_MAX_TOKENS

        if attempt > 0:
            previous_response_excerpt = (last_content or "")[-32768:]

            repair_prompt = (
                prompt
                + "\n\nYour previous response could not be parsed as valid JSON. "
                + "Reason again if needed. After reasoning is complete, output exactly one valid JSON object like {\"response\": \"...\"}. "
                + "Do not wrap the JSON in markdown. Do not add explanations, citations, reasoning traces, or extra text outside the JSON.\n\n"
                + f"Validation error:\n{last_error}\n\n"
                + f"Previous retry info:\n{json.dumps(last_retry_info, ensure_ascii=False)}\n\n"
                + f"Previous response excerpt:\n{previous_response_excerpt}"
            )

            current_messages = [
                {"role": "system", "content": ANSWER_SYSTEM_PROMPT},
                {"role": "user", "content": repair_prompt},
            ]

        response = create_chat_completion_with_optional_schema(
            model=LLM_MODEL_NAME,
            messages=current_messages,
            max_tokens=current_max_tokens,
            **LLM_SAMPLING_KWARGS,
            extra_body=LLM_EXTRA_BODY,
        )

        msg = response.choices[0].message
        content = msg.content or ""
        reasoning = get_reasoning_from_message(msg)

        last_content = content
        last_reasoning = reasoning

        usage_dict = response.usage.model_dump() if response.usage else None
        completion_tokens = get_completion_tokens_from_usage(response.usage)

        try:
            parsed = parse_answer_json(content)

            # If JSON is valid, return immediately.
            # This includes valid {"response": "Information not available"}.
            return {
                "response": parsed["response"],
                "raw_content": content,
                "reasoning_content": reasoning,
                "usage": usage_dict,
                "completion_tokens": completion_tokens,
                "max_tokens_used": current_max_tokens,
                "attempt": attempt,
                "retry_reason": None,
            }

        except Exception as e:
            last_error = repr(e)
            last_retry_info = should_retry_after_parse_error(response, content)

            if attempt >= max_retries:
                break

            continue

    # Last-resort fallback from raw content after all JSON retries failed.
    fallback = normalize_answer_text(strip_think_blocks(last_content or ""))

    # Avoid storing invalid verbose text as answer.
    if "{" in fallback or "}" in fallback or len(fallback.split()) > 30:
        fallback = "Information not available"

    return {
        "response": fallback,
        "raw_content": last_content,
        "reasoning_content": last_reasoning,
        "usage": None,
        "parse_error": last_error,
        "retry_info": last_retry_info,
        "max_tokens_used": ANSWER_RETRY_MAX_TOKENS,
        "attempt": "fallback",
        "retry_reason": "invalid_json_after_retries",
    }

In [ ]:
# cell 11
# Output I/O and resume helpers.

def atomic_write_json(path, data):
    # Atomic JSON write.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def load_existing_answer_map(path):
    # Load existing answers for resume.
    # This is called separately for each ablation output file.
    path = Path(path)

    if not path.exists():
        return {}

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

    if not isinstance(data, list):
        return {}

    out = {}

    for pos, rec in enumerate(data):
        if not isinstance(rec, dict):
            continue

        source_index = rec.get("source_index", pos)

        try:
            source_index = int(source_index)
        except Exception:
            source_index = pos

        out[source_index] = rec

    return out

def save_answer_map(path, answer_map):
    # Save answers sorted by source_index.
    ordered = [
        answer_map[idx]
        for idx in sorted(answer_map.keys())
    ]
    atomic_write_json(path, ordered)

def make_base_output_record(record):
    # Convert phase-2 record to phase-3 output schema.
    return {
        "source_index": int(record.get("source_index", 0)),
        "type": record.get("type"),
        "question": record.get("question"),
        "gt": record.get("answer"),
    }

def make_success_output_record(record, llm_result):
    # Build final answer record.
    out = make_base_output_record(record)
    out["response"] = llm_result["response"]
    out["attempt"] = llm_result.get("attempt")
    out["max_tokens_used"] = llm_result.get("max_tokens_used")
    out["completion_tokens"] = llm_result.get("completion_tokens")
    out["retry_reason"] = llm_result.get("retry_reason")
    return out

def make_error_output_record(record, error):
    # Build error record but keep response field.
    out = make_base_output_record(record)
    out["response"] = "Information not available"
    out["error"] = str(error)
    out["traceback"] = traceback.format_exc()
    return out

print("Output helpers are ready.")
print("Existing answers will be loaded separately for each ablation run.")

Output helpers are ready.
Existing answers will be loaded separately for each ablation run.


In [ ]:
# cell 12
# Run Gemma4 answer generation for all ablations independently.

def run_answer_generation_for_ablation(run_config):
    # Run answer generation for exactly one ablation file.
    # No evidence records or answer map are shared across ablation runs.
    ablation_name = run_config["name"]
    local_evidence_path = run_config["local_evidence_path"]
    drive_answer_path = run_config["drive_answer_path"]

    evidence_records = load_evidence_records(local_evidence_path)
    existing_answers = load_existing_answer_map(drive_answer_path)

    start_idx = int(ANSWER_START_INDEX)
    end_idx = len(evidence_records) if ANSWER_END_INDEX is None else int(ANSWER_END_INDEX)

    run_records = evidence_records[start_idx:end_idx]

    print("=" * 120)
    print("Starting ablation:", ablation_name)
    print("Total evidence records:", len(evidence_records))
    print("Run range:", start_idx, "to", end_idx)
    print("Existing answers:", len(existing_answers))
    print("Output:", drive_answer_path)

    progress = tqdm(
        run_records,
        desc=f"Generating Gemma4 answers [{ablation_name}]",
        dynamic_ncols=True,
    )

    for local_pos, record in enumerate(progress, start=1):
        source_index = int(record.get("source_index", start_idx + local_pos - 1))

        # Skip completed records only for this ablation output file.
        if source_index in existing_answers:
            old = existing_answers[source_index]
            if isinstance(old, dict) and old.get("response") and "error" not in old:
                progress.set_postfix({
                    "ablation": ablation_name,
                    "source_index": source_index,
                    "status": "skipped",
                    "saved": len(existing_answers),
                })
                continue

        try:
            prompt = build_answer_prompt(record)
            llm_result = call_llm_answer(prompt)

            output_record = make_success_output_record(record, llm_result)
            existing_answers[source_index] = output_record
            status = "ok"

        except Exception as e:
            output_record = make_error_output_record(record, e)
            existing_answers[source_index] = output_record
            status = "error"

        # Save after each question.
        if local_pos % SAVE_EVERY_N == 0:
            save_answer_map(drive_answer_path, existing_answers)

        if local_pos % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        progress.set_postfix({
            "ablation": ablation_name,
            "source_index": source_index,
            "status": status,
            "saved": len(existing_answers),
        })

    save_answer_map(drive_answer_path, existing_answers)

    print("Ablation answer generation finished:", ablation_name)
    print("Saved records:", len(existing_answers))
    print("Output file:", drive_answer_path)

    # Clear per-ablation state before the next ablation run.
    del evidence_records
    del run_records
    del existing_answers

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for run_config in ABLATION_RUNS:
    run_answer_generation_for_ablation(run_config)

print("=" * 120)
print("All Gemma4 ablation answer generation finished.")

Starting ablation: without_dense_faiss
Total evidence records: 1000
Run range: 0 to 1000
Existing answers: 0
Output: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/answer/2wikimultihopqa_answer_gemma4.json


Generating Gemma4 answers [without_dense_faiss]:   0%|          | 0/1000 [00:00<?, ?it/s]

Ablation answer generation finished: without_dense_faiss
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/ablation/without_dense_faiss/answer/2wikimultihopqa_answer_gemma4.json
Starting ablation: without_bm25_token
Total evidence records: 1000
Run range: 0 to 1000
Existing answers: 0
Output: /content/drive/MyDrive/final_project/ablation/without_bm25_token/answer/2wikimultihopqa_answer_gemma4.json


Generating Gemma4 answers [without_bm25_token]:   0%|          | 0/1000 [00:00<?, ?it/s]

Ablation answer generation finished: without_bm25_token
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/ablation/without_bm25_token/answer/2wikimultihopqa_answer_gemma4.json
Starting ablation: without_bm25_3char
Total evidence records: 1000
Run range: 0 to 1000
Existing answers: 0
Output: /content/drive/MyDrive/final_project/ablation/without_bm25_3char/answer/2wikimultihopqa_answer_gemma4.json


Generating Gemma4 answers [without_bm25_3char]:   0%|          | 0/1000 [00:00<?, ?it/s]

Ablation answer generation finished: without_bm25_3char
Saved records: 1000
Output file: /content/drive/MyDrive/final_project/ablation/without_bm25_3char/answer/2wikimultihopqa_answer_gemma4.json
All Gemma4 ablation answer generation finished.
